# Session 2: shadow cones and the eclipse limit
**SAFETY.** Never look at the Sun without certified eclipse glasses, even during a partial
eclipse. Everything below is calculation only.

**A. Real-world problem.** Why is a total solar eclipse such a rare sight at any one place?
The Moon's dark shadow is a thin cone whose tip only just reaches the Earth, and the Moon
passes between the Sun and the Earth only when it is close to the plane of the Earth's
orbit. Today we compute the cone, the width of the dark track, and how near a "node" the
Moon must be.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

R_SUN = 696000.0         # km
R_MOON = 1737.0          # km
R_EARTH = 6378.0         # km, equatorial radius
D_SUN = 1.496e8          # km, Sun-Moon distance is taken equal to this (error is under 0.3 percent)
D_MOON = 384400.0        # km, mean
D_MOON_PERIGEE = 356500.0    # km, a close Moon
D_MOON_APOGEE = 406700.0     # km, a far Moon
INCLINATION = 5.145      # degrees, tilt of the Moon's orbit against the ecliptic
SYNODIC = 29.530589      # days, new moon to new moon
DRACONIC_YEAR = 346.62   # days, the Sun returns to the same node
TROPICAL_YEAR = 365.2422   # days

## Step 1: angular sizes (check the "coincidence")
Sun: diameter 1.3927e6 km at 1.496e8 km. Moon: diameter 3474 km at 384400 km.
The angle across a disc is 2 arctan(radius / distance).

In [ ]:
def angular_diameter_deg(diameter_km, distance_km):
    return float(np.degrees(2 * np.arctan(diameter_km / 2 / distance_km)))

## Step 2: the umbra (dark cone) by similar triangles
Draw the Sun (radius R) and Moon (radius r) with a line from the top of the Sun to the
top of the Moon, continued until it meets the axis at the tip, a distance L behind the
Moon. The small triangle (tip to Moon) and the big triangle (tip to Sun) are similar:
r / L = R / (L + D), so r (L + D) = R L and

    L = D r / (R - r)        (D = Sun-Moon distance, L measured from the Moon)

Beyond the tip, the cone opens again: that region is the *antumbra*, where an observer
sees a ring of Sun (annular eclipse). The *penumbra* is the wider cone from the other pair
of tangent lines: its radius grows as r + s (R + r) / D at distance s behind the Moon.

In [ ]:
def umbra_length(sun_radius=R_SUN, moon_radius=R_MOON, sun_moon_distance=D_SUN):
    return float(sun_moon_distance * moon_radius / (sun_radius - moon_radius))


def umbra_radius_at(s, sun_radius=R_SUN, moon_radius=R_MOON, sun_moon_distance=D_SUN):
    """Signed radius (km) of the dark shadow at distance s behind the Moon.
    Positive: inside the umbra (total). Negative: beyond the tip, the antumbra (annular);
    the magnitude is the radius of the antumbra circle."""
    L = umbra_length(sun_radius, moon_radius, sun_moon_distance)
    return float(moon_radius * (L - s) / L)


def penumbra_radius_at(s, sun_radius=R_SUN, moon_radius=R_MOON, sun_moon_distance=D_SUN):
    return float(moon_radius + s * (sun_radius + moon_radius) / sun_moon_distance)

## Step 3: total or annular? the width of the path
The ground nearest the Moon is at distance s = d - R_EARTH (the sub-lunar point).
If L is longer than s the tip is beyond the ground: total. Otherwise annular.
The dark circle on the ground has radius |umbra_radius_at(s)|, so the path is about
twice that wide (the real path is somewhat wider when the Sun is low, because the ground
is tilted against the shadow; this model ignores that).

In [ ]:
def ground_distance(moon_distance):
    return moon_distance - R_EARTH


def eclipse_kind_overhead(moon_distance, sun_moon_distance=D_SUN):
    r = umbra_radius_at(ground_distance(moon_distance), sun_moon_distance=sun_moon_distance)
    return "total" if r > 0 else "annular"


def path_width_km(moon_distance, sun_moon_distance=D_SUN):
    return float(2 * abs(umbra_radius_at(ground_distance(moon_distance), sun_moon_distance=sun_moon_distance)))


def critical_total_distance(sun_moon_distance=D_SUN):
    """Largest Earth-Moon distance (km) for which the tip reaches the sub-lunar ground."""
    return umbra_length(sun_moon_distance=sun_moon_distance) + R_EARTH


def totality_minutes(moon_distance, equator_rotation_kms=0.465):
    """Rough duration (minutes) of totality at the equator: path width / shadow speed over ground."""
    v = 2 * np.pi * moon_distance / (27.321661 * 86400.0) - equator_rotation_kms
    return float(path_width_km(moon_distance) / v / 60.0)


def earth_umbra_radius_at_moon(moon_distance=D_MOON, sun_moon_distance=D_SUN):
    """Radius (km) of the Earth's dark shadow where the Moon crosses it (lunar eclipse)."""
    return float(R_EARTH - moon_distance * (R_SUN - R_EARTH) / sun_moon_distance)

## Step 4: the eclipse limit (why not every month?)
The Moon's orbit is tilted by i = 5.145 degrees. Seen from the Earth the Moon's ecliptic
latitude is  sin(beta) = sin(i) sin(u), where u is the angle along the orbit measured from
the ascending node (the point where the Moon crosses the ecliptic going north).
At new moon the Moon is in the Sun's direction, so the Sun-Moon separation equals |beta|.

A partial eclipse is visible somewhere if the shadow's axis passes within R_EARTH plus the
penumbra radius of the Earth's centre. In angles this is

    |beta| < pi + s_sun + s_moon        (pi = parallax of the Moon = arcsin(R_EARTH / d))

(s = angular radii of the Sun and the Moon). A *central* eclipse (the axis hits the Earth's
disc) needs only |beta| < pi. Solving sin(beta) = sin(i) sin(u) for u:

    u_limit = arcsin( sin(beta_limit) / sin(i) )

In [ ]:
def parallax_deg(moon_distance):
    return float(np.degrees(np.arcsin(R_EARTH / moon_distance)))


def beta_limit_partial(moon_distance, sun_distance=D_SUN):
    s_sun = np.degrees(np.arctan(R_SUN / sun_distance))
    s_moon = np.degrees(np.arctan(R_MOON / moon_distance))
    return float(parallax_deg(moon_distance) + s_sun + s_moon)


def beta_limit_central(moon_distance):
    return parallax_deg(moon_distance)


def node_limit_deg(beta_limit, inclination=INCLINATION):
    """Largest angle from the node (degrees) at which the Moon can still be within beta_limit."""
    ratio = np.sin(np.radians(beta_limit)) / np.sin(np.radians(inclination))
    if ratio >= 1.0:          # the orbit never gets that far from the ecliptic: every angle qualifies
        return 90.0
    return float(np.degrees(np.arcsin(ratio)))


def limit_range():
    """(largest partial limit, certain partial limit, largest central limit), degrees from the node.
    Largest: Moon at perigee 356400 km, Sun at perihelion 1.471e8 km, orbit tilt 5.0 degrees.
    Certain: Moon at apogee 406700 km, Sun at aphelion 1.521e8 km, tilt 5.3 degrees."""
    big = node_limit_deg(beta_limit_partial(356400.0, 1.471e8), 5.0)
    sure = node_limit_deg(beta_limit_partial(406700.0, 1.521e8), 5.3)
    central = node_limit_deg(beta_limit_central(356400.0), 5.0)
    return big, sure, central


def node_limit_numeric(beta_limit, inclination=INCLINATION):
    """Same limit by brute force: move a Moon along its tilted orbit, build 3-D unit vectors,
    put the Sun at the Moon's ecliptic longitude (new moon) and bisect on the separation."""
    inc = np.radians(inclination)

    def separation(u):
        moon = np.array([np.cos(u), np.cos(inc) * np.sin(u), np.sin(inc) * np.sin(u)])
        lon = np.arctan2(moon[1], moon[0])
        sun = np.array([np.cos(lon), np.sin(lon), 0.0])
        return np.degrees(np.arccos(np.clip(moon @ sun, -1.0, 1.0)))

    lo, hi = 0.0, np.pi / 2
    for _ in range(60):
        mid = (lo + hi) / 2
        if separation(mid) < beta_limit:
            lo = mid
        else:
            hi = mid
    return float(np.degrees(lo))

## Step 5: eclipse seasons (a design problem)
The nodes drift slowly, so the Sun returns to the same node every 346.62 days (the
draconic year), not every 365.24 days. The Sun passes a node twice per draconic year:
eclipse seasons are 346.62 / 2 = 173.31 days apart. The Sun moves along the ecliptic,
relative to the node, at 360 / 346.62 degrees per day, so a season in which the Sun stays
within +/- limit of a node lasts 2 limit / (that rate) days.

In [ ]:
def season_spacing_days():
    return DRACONIC_YEAR / 2.0


def sun_node_rate_deg_per_day():
    return 360.0 / DRACONIC_YEAR


def season_length_days(limit_deg):
    return float(2 * limit_deg / sun_node_rate_deg_per_day())


def seasons_per_tropical_year():
    return TROPICAL_YEAR / season_spacing_days()


def seasons_drift_days_per_year():
    """How many days earlier each year's eclipse seasons arrive."""
    return TROPICAL_YEAR - DRACONIC_YEAR

## Step 6: see it
Left: signed shadow radius on the ground against the Moon's distance. Right: the limit.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print(f"Sun {angular_diameter_deg(1.3927e6, 1.496e8):.3f} deg, Moon {angular_diameter_deg(3474, 384400):.3f} deg")
    L = umbra_length()
    print(f"umbra length L = {L:.0f} km; mean ground distance {ground_distance(D_MOON):.0f} km")
    print(f"total if Earth-Moon distance below {critical_total_distance():.0f} km")
    for name, d in (("perigee", D_MOON_PERIGEE), ("mean", D_MOON), ("apogee", D_MOON_APOGEE)):
        print(f"{name:8s} d={d:.0f}: {eclipse_kind_overhead(d):8s} path width {path_width_km(d):6.1f} km")
    print(f"penumbra radius at the Earth {penumbra_radius_at(D_MOON):.0f} km")
    print(f"limits (mean Moon): partial {node_limit_deg(beta_limit_partial(D_MOON)):.1f} deg, "
          f"central {node_limit_deg(beta_limit_central(D_MOON)):.1f} deg")

    d = np.linspace(354000, 410000, 200)
    r = [umbra_radius_at(ground_distance(x)) for x in d]
    fig, ax = plt.subplots(figsize=(6.4, 4))
    ax.plot(d / 1000, r, color="tab:blue")
    ax.axhline(0, color="gray", lw=0.8)
    ax.axvline(critical_total_distance() / 1000, color="tab:red", ls=":")
    ax.set_xlabel("Earth-Moon distance (thousand km)")
    ax.set_ylabel("shadow radius on the ground (km); negative = ring")
    ax.set_title("Total (above 0) or annular (below 0)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "umbra_on_ground.png", dpi=120)

    u = np.linspace(-30, 30, 400)
    beta = np.degrees(np.arcsin(np.sin(np.radians(INCLINATION)) * np.sin(np.radians(u))))
    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    ax2.plot(u, beta, color="tab:blue", label="Moon's ecliptic latitude")
    bp, bc = beta_limit_partial(D_MOON), beta_limit_central(D_MOON)
    for b, name, c in ((bp, "partial limit", "tab:orange"), (bc, "central limit", "tab:green")):
        ax2.axhline(b, color=c, ls="--")
        ax2.axhline(-b, color=c, ls="--", label=name)
    ax2.axvspan(-node_limit_deg(bp), node_limit_deg(bp), color="gold", alpha=0.15)
    ax2.set_xlabel("angle from the node (degrees)")
    ax2.set_ylabel("Moon's ecliptic latitude (degrees)")
    ax2.set_title("An eclipse needs the Moon near a node")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "eclipse_limit.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Compute the umbra length for a Sun-Moon distance of 1.52e8 km and say whether the mean Earth-Moon distance gets any closer to total.
2. **Stretch.** Use `node_limit_numeric` and `node_limit_deg` for beta limits of 1.0 and 1.5 degrees and compare the two answers.
3. **Explain.** Explain why the node limit for a central eclipse (10.7 degrees) is smaller than the limit for a partial eclipse (16.7 degrees).

## Assessment
- **A. Conceptual.** At the mean Moon distance the discs look almost equal. Why is an eclipse at the mean distance annular, not total?
- **B. Numerical.** Compute the umbra length for R = 696000 km, r = 1737 km, D = 1.496e8 km and compare it with 384400 km.
- **C. Graph and simulation interpretation.** On the graph of dark-shadow radius against Earth-Moon distance, where does the curve cross zero, and what does a negative value mean?
- **D. Design and optimisation challenge.** Design: how large a limit angle must we assume so that every eclipse season is certain to contain at least one new moon within the limit? Compare 15.4 degrees with the synodic month.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("angular diameters: Sun 0.533, Moon 0.518 degrees",
                   abs(L.angular_diameter_deg(1.3927e6, 1.496e8) - 0.533) < 0.001
                   and abs(L.angular_diameter_deg(3474, 384400) - 0.518) < 0.001))

    Lu = L.umbra_length()
    checks.append(("hand calculation: L = D r / (R - r) = 1.496e8 * 1737 / 694263 = 374289 km",
                   abs(1.496e8 * 1737 / (696000 - 1737) - 374289) < 1 and abs(Lu - 374289) < 1))
    checks.append(("umbra length 374289 km is 10111 km short of the mean Moon-Earth distance 384400 km",
                   abs((384400 - Lu) - 10111) < 1))
    # similar triangles, independent check: the line through the Sun's and Moon's edges reaches the axis at L
    slope = (L.R_SUN - L.R_MOON) / L.D_SUN
    checks.append(("similar triangles: the tangent line has slope (R - r)/D and crosses the axis at r/slope",
                   abs(L.R_MOON / slope - Lu) < 1e-6 and abs(L.umbra_radius_at(0.0) - L.R_MOON) < 1e-9
                   and abs(L.umbra_radius_at(Lu)) < 1e-9))

    checks.append(("nearest ground is 378022 km from the mean Moon, so the tip stops 3733 km short; antumbra radius 17.3 km",
                   L.ground_distance(384400.0) == 378022.0 and abs(Lu - 378022 + 3733) < 1
                   and abs(abs(L.umbra_radius_at(378022.0)) - 17.3) < 0.1))
    v = 2 * np.pi * 356500.0 / (27.321661 * 86400.0) - 0.465
    checks.append(("shadow speed at perigee is 0.484 km/s, and 224.3 km / 0.484 km/s is 463 s",
                   abs(v - 0.484) < 0.001 and abs(L.path_width_km(356500.0) / v - 463) < 1.5))
    checks.append(("total if the Earth-Moon distance is below 380667 km (L + R_EARTH)",
                   abs(L.critical_total_distance() - 380667) < 1))
    checks.append(("mean Moon (384400 km) gives annular; perigee 356500 km total; apogee 406700 km annular",
                   L.eclipse_kind_overhead(384400.0) == "annular" and L.eclipse_kind_overhead(356500.0) == "total"
                   and L.eclipse_kind_overhead(406700.0) == "annular"))
    s = 356500.0 - 6378.0
    hand = 2 * 1737.0 * (374289.0 - s) / 374289.0
    checks.append((f"worked example: perigee totality path is about 224 km wide (hand {hand:.1f} km)",
                   abs(L.path_width_km(356500.0) - 224.3) < 0.2 and abs(hand - 224.3) < 0.2))
    checks.append(("path width stays below 300 km for every Moon distance from 356000 to 407000 km",
                   max(L.path_width_km(d) for d in np.linspace(356000, 407000, 60)) < 300))
    checks.append(("mean Moon: the ring (antumbra) circle is only 34.6 km wide; apogee 241.6 km",
                   abs(L.path_width_km(384400.0) - 34.6) < 0.2 and abs(L.path_width_km(406700.0) - 241.6) < 0.2))
    checks.append(("totality lasts about 7.7 minutes at perigee on the equator (224 km / 0.484 km/s)",
                   abs(L.totality_minutes(356500.0) - 7.7) < 0.1))
    checks.append(("penumbra radius at the ground is 3500 km, so the partial region is about 7000 km wide",
                   abs(L.penumbra_radius_at(384400.0 - 6378.0) - 3500) < 2))
    checks.append(("lunar eclipse: Earth's shadow at the Moon is 4606 km in radius, 2.65 Moon radii",
                   abs(L.earth_umbra_radius_at_moon() - 4606) < 2 and abs(L.earth_umbra_radius_at_moon() / 1737 - 2.65) < 0.01))

    # eclipse limit
    bp, bc = L.beta_limit_partial(384400.0), L.beta_limit_central(384400.0)
    checks.append(("mean Moon: parallax 0.951; latitude limits 1.476 (partial) and 0.951 degrees (central)",
                   abs(L.parallax_deg(384400.0) - 0.951) < 0.001 and abs(bp - 1.476) < 0.001 and abs(bc - 0.951) < 0.001))
    hand = np.degrees(np.arcsin(np.sin(np.radians(1.4762)) / np.sin(np.radians(5.145))))
    checks.append((f"hand calculation: asin(sin 1.4762 / sin 5.145) = {hand:.1f} degrees",
                   abs(hand - 16.7) < 0.05 and abs(L.node_limit_deg(bp) - 16.7) < 0.05))
    checks.append(("limits from the node: partial about 16.7 degrees, central about 10.7 degrees (mean Moon)",
                   abs(L.node_limit_deg(bp) - 16.7) < 0.05 and abs(L.node_limit_deg(bc) - 10.7) < 0.05))
    checks.append(("brute-force 3-D vectors agree with the formula to 0.01 degrees",
                   abs(L.node_limit_numeric(bp) - L.node_limit_deg(bp)) < 0.01
                   and abs(L.node_limit_numeric(bc) - L.node_limit_deg(bc)) < 0.01))
    big, sure, cen = L.limit_range()
    checks.append(("extremes: partial possible up to 18.4, certain within 15.4; central up to 11.8 degrees",
                   abs(big - 18.4) < 0.05 and abs(sure - 15.4) < 0.05 and abs(cen - 11.8) < 0.05))
    checks.append(("edge: a latitude limit larger than the orbit tilt allows every angle (90 degrees)",
                   L.node_limit_deg(10.0) == 90.0))

    # seasons
    checks.append(("eclipse seasons are 173.31 days apart; the Sun passes the node at 1.0386 degrees per day",
                   abs(L.season_spacing_days() - 173.31) < 1e-9 and abs(L.sun_node_rate_deg_per_day() - 1.0386) < 1e-4))
    checks.append(("a season of limit 16.7 degrees lasts 32.1 days, longer than a synodic month 29.53 days",
                   abs(L.season_length_days(16.7) - 32.1) < 0.1 and L.season_length_days(16.7) > L.SYNODIC))
    checks.append(("with the certain limit 15.4 degrees the season is 29.65 days: still longer than 29.53, so every season has a new moon in it",
                   abs(L.season_length_days(15.4) - 29.65) < 0.02 and L.season_length_days(15.4) > L.SYNODIC))
    checks.append(("2.107 seasons per year; seasons arrive 18.62 days earlier each year; about 21 in 10 years",
                   abs(L.seasons_per_tropical_year() - 2.107) < 0.001 and abs(L.seasons_drift_days_per_year() - 18.62) < 0.01
                   and round(10 * L.seasons_per_tropical_year()) == 21))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
